# Fleet Dispatch & Charging Agent — Demo

Notebook version of `demo.py`'s fixed, deterministic scenario (3 vehicles, 4 hand-placed orders, 1 scripted breakdown — see `demo.py` for the exact input), with the map visualization from `map_export.py` rendered inline instead of opened as a separate HTML file.

Runs the scenario **once** below and reuses that single run for both the text output and the map, so this doesn't cost two rounds of Gemini API calls if `GEMINI_API_KEY` is configured in `.env`.

In [1]:
import sys
from pathlib import Path

# demo.py, sim.py, map_export.py, map_screenshot.py live one directory up
# from this notebook (fleet_dispatch_baseline/), not alongside it.
sys.path.insert(0, str(Path.cwd().parent))

from demo import run_scenario
from sim import render_grid

sim, frames = run_scenario(record_frames=True)

print("-- Event log --")
print("\n".join(sim.log))
print()
print("-- Final grid --")
print(render_grid(sim))
print()

delivered = [o for o in sim.orders.values() if o.delivered_tick is not None]
on_time = [o for o in delivered if o.delivered_tick <= o.deadline]
print(f"orders: {len(sim.orders)} created, {len(delivered)} delivered, {len(on_time)} on time")

-- Event log --
t=  0  order  0 created  -> dest (4, 0), deadline t=12
t=  0    llm batch dispatch unavailable: google-genai package not installed
t=  0  order  0 dispatched -> vehicle 0 (dist 4, battery 100%, picked by rule)
t=  2  order  2 created  -> dest (0, 6), deadline t=14
t=  2    llm batch dispatch unavailable: google-genai package not installed
t=  2  order  2 dispatched -> vehicle 1 (dist 6, battery 100%, picked by rule)
t=  5  order  5 created  -> dest (3, 3), deadline t=17
t=  5  order  0 delivered by vehicle 0 at (4, 0) -- ON TIME (deadline t=12)
t=  5    llm batch dispatch unavailable: google-genai package not installed
t=  5  order  5 dispatched -> vehicle 2 (dist 6, battery 100%, picked by rule)
t=  6  INCIDENT: vehicle 1 broke down at (0, 4) (order 2 needs redispatch)
t=  9  order  9 created  -> dest (6, 2), deadline t=21
t=  9  order  5 delivered by vehicle 2 at (3, 3) -- ON TIME (deadline t=17)
t= 10    llm batch dispatch unavailable: google-genai package not instal

## Map visualization

Same Leaflet/OpenStreetMap page `map_export.py` writes to `map_demo.html` — reused here directly (`TEMPLATE` + `build_data()`, no duplicated map logic) and rendered inline via an `<iframe srcdoc=...>` instead of opened in a separate browser tab. Play/pause and the tick scrubber both work inline.

In [2]:
import html
import json

from IPython.display import HTML

from map_export import TEMPLATE, build_data

page_html = TEMPLATE.replace("__DATA__", json.dumps(build_data(frames)))
HTML(f'<iframe srcdoc="{html.escape(page_html)}" width="100%" height="620" style="border:1px solid #444;"></iframe>')

/users/samarthms/conda/miniconda3/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
